!pip install pdfplumber pandas

In [1]:
!pip install pdfplumber pandas
!pip install pypdfium2 pytesseract Pillow pandas
!apt-get install -y tesseract-ocr tesseract-ocr-fra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.3 MB/s eta 0:00:00:00:01
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-fra
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 527 kB of archives.
After this operation, 1,145 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-fra all 1:4.00~git30-7274cfa-1.1 [527 kB]
Fetched 527 kB in 0s (11.8 MB/s)      
Selecting previously unselected package tesserac

## Importation des bibliothèques

In [2]:
import re
import pandas as pd
import numpy as np
import pytesseract
import pypdfium2 as pdfium
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [3]:
# Pages choisies (index 0-based) :
# 17,18 = pages 18-19 (tableau délits + points)
# 26-30 = pages 27-31 (délits + amendes)
# 31,32,33 = pages 32-34 (contraventions + amendes)

PAGES_A_EXTRAIRE = [17, 18, 26, 27, 28, 29, 30, 31, 32, 33]

PDF_PATH = "/kaggle/input/datasets/juniorbambara/code-de-la-route-fr/code de la route fr.pdf"  

pdf = pdfium.PdfDocument(PDF_PATH)
raw_text = ""

for page_idx in PAGES_A_EXTRAIRE:
    page = pdf[page_idx]
    bitmap = page.render(scale=2.5)  # scale 2.5 = bonne qualité OCR
    img = bitmap.to_pil()
    text = pytesseract.image_to_string(img, lang="fra")
    raw_text += f"\n\n--- PAGE {page_idx+1} ---\n\n" + text
    print(f"Page {page_idx+1} : {len(text)} caractères extraits")

print(f"\nTotal : {len(raw_text)} caractères")
print("\nAperçu :\n", raw_text[:500])

Page 18 : 5915 caractères extraits
Page 19 : 5314 caractères extraits
Page 27 : 5969 caractères extraits
Page 28 : 5560 caractères extraits
Page 29 : 5529 caractères extraits
Page 30 : 5110 caractères extraits
Page 31 : 5728 caractères extraits
Page 32 : 5341 caractères extraits
Page 33 : 5278 caractères extraits
Page 34 : 5426 caractères extraits

Total : 55360 caractères

Aperçu :
 

--- PAGE 18 ---

1660

N° 5874 — 7 chaoual 1431 (16-9-2010)

 

LIVRE DEUX
DES SANCTIONS ET DE LA PROCEDURE
TITRE PREMIER
DES SANCTIONS ET DES MESURES ADMINISTRATIVES
Chapitre premier
De la suspension et du retrait administratifs
du permis de conduire

Article 95

L'administration prononce la suspension du permis de
conduire, si la personne qui en est titulaire n'a pas acquitté le
montant de l’amende prononcée à son encontre par décision
judiciaire ayant acquis la force de la chose jugée ou pa


In [4]:
def normalize_text(text):
    # Supprimer les numéros de page du Bulletin Officiel
    text = re.sub(r'\d{4}\n+N°\s*5874[^\n]*\n', '', text)
    # Supprimer les headers répétitifs
    text = re.sub(r'BULLETIN OFFICIEL[^\n]*\n', '', text)
    # Nettoyer les espaces multiples
    text = re.sub(r'[ \t]+', ' ', text)
    # Normaliser les sauts de ligne
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Corriger erreurs OCR fréquentes sur ce doc
    text = text.replace('1 000', '1.000')
    text = text.replace('1 400', '1.400')
    text = text.replace('| ', '')
    return text.strip()

raw_text_clean = normalize_text(raw_text)
print("Avant :", len(raw_text), "caractères")
print("Après :", len(raw_text_clean), "caractères")
print("\nAperçu :\n", raw_text_clean[:500])

Avant : 55360 caractères
Après : 55211 caractères

Aperçu :
 --- PAGE 18 ---

 

LIVRE DEUX
DES SANCTIONS ET DE LA PROCEDURE
TITRE PREMIER
DES SANCTIONS ET DES MESURES ADMINISTRATIVES
Chapitre premier
De la suspension et du retrait administratifs
du permis de conduire

Article 95

L'administration prononce la suspension du permis de
conduire, si la personne qui en est titulaire n'a pas acquitté le
montant de l’amende prononcée à son encontre par décision
judiciaire ayant acquis la force de la chose jugée ou par décision
administrative et/ou n’a pas payé l


In [5]:
def segmenter_articles(text):
    # Split sur "Article" suivi d'un numéro ou "premier"
    pattern = r'(?=Article\s+(?:\d+|premier))'
    segments = re.split(pattern, text, flags=re.IGNORECASE)
    
    articles = []
    for seg in segments:
        seg = seg.strip()
        if len(seg) < 20:
            continue
        # Extraire le numéro d'article
        match = re.match(r'Article\s+(premier|\d+)', seg, re.IGNORECASE)
        if match:
            num = match.group(1)
            if num.lower() == 'premier':
                num = '1'
            contenu = seg[match.end():].strip()
            articles.append({'article_id': int(num), 'contenu': contenu})
    
    return articles

articles = segmenter_articles(raw_text_clean)
print(f"Nombre d'articles segmentés : {len(articles)}")
print("\nExemple article 184 :")
for a in articles:
    if a['article_id'] == 184:
        print(a['contenu'][:300])
        break

Nombre d'articles segmentés : 86

Exemple article 184 :
Est punie d’une amende de sept cents (700) à mille quatre
cents (1.400) dirhams, toute personne qui a commis une
infraction de la première classe.

Est considérée infraction de la première classe, l’une des
infractions suivantes :

1) le dépassement de la vitesse de 30 à moins de 50 km/h
au-dessus d


In [6]:
def extraire_amendes(texte):
    """Extrait amende min et max en dirhams"""
    # Pattern : "X (1.200) à Y (5.000) dirhams" ou "X (700) dirhams"
    pattern = r'(?:de\s+)?(?:\w+\s+){0,4}\((\d[\d.]*)\)\s+à\s+(?:\w+\s+){0,4}\((\d[\d.]*)\)\s+dirhams'
    match = re.search(pattern, texte, re.IGNORECASE)
    if match:
        return int(match.group(1).replace('.', '')), int(match.group(2).replace('.', ''))
    # Amende fixe unique
    pattern2 = r'\((\d[\d.]*)\)\s+dirhams'
    match2 = re.search(pattern2, texte, re.IGNORECASE)
    if match2:
        val = int(match2.group(1).replace('.', ''))
        return val, val
    return None, None

def extraire_points(texte):
    """Extrait points de retrait"""
    match = re.search(r'(\d+)\s+points?\s+(?:à retirer|de retrait|retrait)', texte, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def extraire_classe(texte):
    """Extrait classe d'infraction"""
    if re.search(r'premi[eè]re\s+classe', texte, re.IGNORECASE):
        return 1
    if re.search(r'deuxi[eè]me\s+classe', texte, re.IGNORECASE):
        return 2
    if re.search(r'troisi[eè]me\s+classe', texte, re.IGNORECASE):
        return 3
    if re.search(r'd[ée]lit', texte, re.IGNORECASE):
        return 'délit'
    return None

def extraire_description(texte):
    """Première phrase significative = description de l'infraction"""
    # Nettoyer sauts de ligne
    texte_clean = re.sub(r'\n+', ' ', texte).strip()
    # Prendre jusqu'au premier point ou 300 chars
    match = re.match(r'(.{20,300}?[.;])', texte_clean)
    if match:
        return match.group(1).strip()
    return texte_clean[:300].strip()

def extraire_mots_cles(texte):
    """Extrait mots-clés thématiques"""
    keywords = {
        'vitesse': r'vitesse|km/h',
        'alcool': r'alcool|ivresse|stupéfiant',
        'stationnement': r'stationnement|stationn',
        'feux': r'feu rouge|signalisation',
        'permis': r'permis de conduire',
        'poids_lourd': r'poids.{0,5}lourd|3[.,]500|marchandises',
        'autoroute': r'autoroute',
        'accident': r'accident|blessure|homicide',
        'ceinture': r'ceinture',
        'téléphone': r'téléphon|appareil',
        'priorité': r'priorité',
        'dépassement': r'dépassement',
    }
    found = []
    for kw, pattern in keywords.items():
        if re.search(pattern, texte, re.IGNORECASE):
            found.append(kw)
    return ', '.join(found) if found else 'général'

def extraire_categorie_vehicule(texte):
    """Extrait catégorie de véhicule mentionnée"""
    if re.search(r'moto|motocycle|tricycle', texte, re.IGNORECASE):
        return 'moto'
    if re.search(r'poids.{0,5}lourd|3[.,]500|marchandises|ensemble de véhicules', texte, re.IGNORECASE):
        return 'poids_lourd'
    if re.search(r'transport.{0,10}commun|autobus|personnes', texte, re.IGNORECASE):
        return 'transport_commun'
    return 'tous'

def extraire_type_infraction(texte):
    if re.search(r'd[ée]lit', texte, re.IGNORECASE):
        return 'délit'
    if re.search(r'contravention', texte, re.IGNORECASE):
        return 'contravention'
    if re.search(r'emprisonnement', texte, re.IGNORECASE):
        return 'délit_pénal'
    return 'infraction'

def extraire_gravite(amende_max, classe):
    if amende_max and amende_max >= 10000:
        return 'élevée'
    if amende_max and amende_max >= 1000:
        return 'moyenne'
    if classe == 'délit':
        return 'élevée'
    return 'faible'

# Test rapide
a = articles[0]
print("Article:", a['article_id'])
print("Amende:", extraire_amendes(a['contenu']))
print("Points:", extraire_points(a['contenu']))
print("Classe:", extraire_classe(a['contenu']))
print("Mots-clés:", extraire_mots_cles(a['contenu']))



Article: 95
Amende: (None, None)
Points: None
Classe: None
Mots-clés: général


In [7]:
# Test sur article 184 qui doit avoir une amende
for a in articles:
    if a['article_id'] == 184:
        print("Contenu brut :\n", a['contenu'][:400])
        print("\nAmende:", extraire_amendes(a['contenu']))
        print("Classe:", extraire_classe(a['contenu']))
        print("Mots-clés:", extraire_mots_cles(a['contenu']))
        break

Contenu brut :
 Est punie d’une amende de sept cents (700) à mille quatre
cents (1.400) dirhams, toute personne qui a commis une
infraction de la première classe.

Est considérée infraction de la première classe, l’une des
infractions suivantes :

1) le dépassement de la vitesse de 30 à moins de 50 km/h
au-dessus de la vitesse maximale autorisée, pour tous les
conducteurs ;

2) la circulation sur la voie publique

Amende: (700, 1400)
Classe: 1
Mots-clés: vitesse, stationnement, feux, autoroute, accident, ceinture, dépassement


In [8]:
rows = []

for a in articles:
    texte = a['contenu']
    amende_min, amende_max = extraire_amendes(texte)
    classe = extraire_classe(texte)
    
    rows.append({
        'article_id':          a['article_id'],
        'infraction_desc':     extraire_description(texte),
        'categorie_vehicule':  extraire_categorie_vehicule(texte),
        'amende_min':          amende_min,
        'amende_max':          amende_max,
        'points_retrait':      extraire_points(texte),
        'classe_infraction':   classe,
        'type_infraction':     extraire_type_infraction(texte),
        'niveau_gravite':      extraire_gravite(amende_max, classe),
        'mots_cles':           extraire_mots_cles(texte),
    })

df = pd.DataFrame(rows)

print(f"Shape : {df.shape}")
print(f"\nArticles avec amende : {df['amende_min'].notna().sum()}")
print(f"Articles avec points : {df['points_retrait'].notna().sum()}")
print(f"\nAperçu :")
print(df[df['amende_min'].notna()][['article_id','amende_min','amende_max','classe_infraction','mots_cles']].head(10))

Shape : (86, 10)

Articles avec amende : 34
Articles avec points : 0

Aperçu :
    article_id  amende_min  amende_max classe_infraction  \
17         149      2000.0      4000.0              None   
21         361      5000.0     20000.0              None   
22         151      2000.0      5000.0              None   
23         152      2000.0      8000.0              None   
26         154      1200.0      5000.0              None   
27         155      4000.0     10000.0             délit   
28         156     15000.0     30000.0              None   
29         157      5000.0     30000.0              None   
30         158      5000.0     30000.0              None   
31         159      5000.0     30000.0              None   

                mots_cles  
17                 permis  
21                 permis  
22                 permis  
23                 permis  
26                 permis  
27                 permis  
28                général  
29                général  
30  poid

## Extraction par règles (Regex)

In [9]:
page = pdf[18]  # index 18 = page 19
bitmap = page.render(scale=4.0)  # scale 4 au lieu de 2.5
img = bitmap.to_pil()

text_p19 = pytesseract.image_to_string(img, lang="fra")
print(f"Caractères extraits : {len(text_p19)}")
print(text_p19[:2000])

Caractères extraits : 5163
N° 5874 — 7 chaoual 1431 (16-9-2010)

1661

 

 

POINTS
RETIRER

LE DELIT

Conduite d'un véhicule, dont la conduite
nécessite l’obtention d’un permis de conduire,
malgré une suspension administrative ou
judiciaire du permis de conduire.

Conduite d'un véhicule, dont la conduite nécessite
l'obtention d'un permis de conduire, pendant la
durée de la rétention du permis de conduire,

l Le fait de ne pas déposer un permis de
conduire suspendu. 4

13 Conducteur, sommé de s'arrêter, a refusé de
s'exécuter ou de se soumettre aux
vérifications prescrites ou ne respecte pas
l’ordre d'immobilisation du véhicule ou
refuse de conduire ou de faire conduire son
véhicule en fourrière ou refuse d'obtempérer
aux injonctions légales qui lui sont faites.

Le dépassement de la vitesse de 50 Km/h ou
plus, au dessus de la vitesse maximale
autorisée.

La marche arrière ou le demi-tour sur une
autoroute notamment ou une roule expresse

 

en traversant la bande centrale séparative d

In [12]:
vectorizer = TfidfVectorizer(max_features=50)
X = vectorizer.fit_transform(df['infraction_desc'])

kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

print("Shape TF-IDF:", X.shape)
print("\nDistribution des clusters:")
print(df['cluster'].value_counts())

Shape TF-IDF: (86, 50)

Distribution des clusters:
cluster
1    29
2    19
4    16
3    12
0    10
Name: count, dtype: int64


## Clustering des infractions

- KMeans est l’algorithme de clustering non supervisé qui regroupe les données en K groupes (clusters).

L’idée est de :

- choisir K centres (appelés centroïdes)
- affecter chaque point au centre le plus proche
- recalculer les centres
- répéter jusqu’à stabilisation

- Random state : 
K-Means commence par choisir des centres initiaux au hasard. Sans fixer cette valeur, les résultats peuvent changer à chaque exécution

In [13]:
# Récupérer les clusters
labels = kmeans.labels_
print(labels)

[1 3 2 2 1 2 2 2 2 1 2 2 1 0 2 1 2 4 2 0 2 4 4 4 3 2 0 4 4 0 0 0 0 0 4 1 4
 2 1 4 1 3 1 1 1 3 1 3 1 3 1 2 1 3 1 3 1 1 1 1 3 2 2 1 3 1 3 2 4 4 0 4 1 1
 4 1 0 4 4 3 1 2 4 1 1 1]


In [14]:
df.to_csv('export_final.csv', index=False, encoding='utf-8-sig')

print("export_final.csv généré !")
print(f"Shape : {df.shape}")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nAperçu des articles avec amendes :")
print(df[df['amende_min'].notna()][['article_id','amende_min','amende_max','classe_infraction','type_infraction','niveau_gravite','mots_cles']].head(10).to_string())

export_final.csv généré !
Shape : (86, 11)

Colonnes : ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_min', 'amende_max', 'points_retrait', 'classe_infraction', 'type_infraction', 'niveau_gravite', 'mots_cles', 'cluster']

Aperçu des articles avec amendes :
    article_id  amende_min  amende_max classe_infraction type_infraction niveau_gravite              mots_cles
17         149      2000.0      4000.0              None      infraction        moyenne                 permis
21         361      5000.0     20000.0              None     délit_pénal         élevée                 permis
22         151      2000.0      5000.0              None     délit_pénal        moyenne                 permis
23         152      2000.0      8000.0              None      infraction        moyenne                 permis
26         154      1200.0      5000.0              None     délit_pénal        moyenne                 permis
27         155      4000.0     10000.0             délit   

## Approche modèle pré-entraîné (à compléter)

In [15]:
"""
PARTIE A COMPLETER

Utiliser :
- spaCy
ou
- AraBERT
ou
- CAMeL Tools

Exemple attendu :
1. tokenization
2. embeddings
3. classification
4. NER
"""

'\nPARTIE A COMPLETER\n\nUtiliser :\n- spaCy\nou\n- AraBERT\nou\n- CAMeL Tools\n\nExemple attendu :\n1. tokenization\n2. embeddings\n3. classification\n4. NER\n'

In [16]:
!pip install spacy
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 69.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [18]:
import spacy

nlp = spacy.load("fr_core_news_sm")

# Test sur quelques articles avec amendes - FIXED
sample = df[df['amende_min'].notna()]['infraction_desc'].head(5).tolist()

for text in sample:
    doc = nlp(text)
    print("TEXTE:", text[:100])
    print("TOKENS:", [token.text for token in doc if not token.is_stop and not token.is_punct][:10])
    print("ENTITÉS:", [(ent.text, ent.label_) for ent in doc.ents])
    print("---")

TEXTE: ci-dessous, est punie d'une amende de deux mille (2.
TOKENS: ['ci-dessous', 'punie', 'amende', '2']
ENTITÉS: []
---
TEXTE: du code pénal et sans préjudice de peines plus sévères, est punie d'un emprisonnement de un (1) mois
TOKENS: ['code', 'pénal', 'préjudice', 'peines', 'sévères', 'punie', 'emprisonnement', '1', 'mois', '6']
ENTITÉS: []
---
TEXTE: Est punie d’un emprisonnement de six (6) mois à trois (3) ans et d'une amende de deux mille (2.
TOKENS: ['punie', 'emprisonnement', '6', 'mois', '3', 'ans', 'amende', '2']
ENTITÉS: []
---
TEXTE: Est punie d'une amende de deux mille (2.
TOKENS: ['punie', 'amende', '2']
ENTITÉS: []
---
TEXTE: Toute personne qui conduit avec un faux permis de conduire un véhicule dont la conduite nécessite Po
TOKENS: ['conduit', 'faux', 'permis', 'conduire', 'véhicule', 'conduite', 'nécessite', 'Pobtention', 'permis', 'conduire']
ENTITÉS: [('Pobtention d’', 'MISC'), ('d’', 'MISC')]
---


In [19]:
from spacy.pipeline import EntityRuler

# Ajouter un EntityRuler custom avant le NER de spaCy
ruler = nlp.add_pipe("entity_ruler", before="ner")

patterns = [
    # Montants
    {"label": "AMENDE", "pattern": [{"TEXT": {"REGEX": r"\d[\d.]*"}}, {"LOWER": "dirhams"}]},
    {"label": "AMENDE", "pattern": [{"LOWER": "amende"}, {"LOWER": "de"}]},
    # Emprisonnement
    {"label": "PEINE", "pattern": [{"LOWER": "emprisonnement"}]},
    {"label": "PEINE", "pattern": [{"LOWER": "suspension"}, {"LOWER": "du"}, {"LOWER": "permis"}]},
    # Infractions
    {"label": "INFRACTION", "pattern": [{"LOWER": "dépassement"}, {"LOWER": "de"}, {"LOWER": "vitesse"}]},
    {"label": "INFRACTION", "pattern": [{"LOWER": "conduite"}, {"LOWER": "en"}, {"LOWER": "état"}, {"LOWER": "d'ivresse"}]},
    {"label": "INFRACTION", "pattern": [{"LOWER": "stationnement"}, {"LOWER": "non"}, {"LOWER": "réglementaire"}]},
    {"label": "INFRACTION", "pattern": [{"LOWER": "délit"}, {"LOWER": "de"}, {"LOWER": "fuite"}]},
    # Véhicules
    {"label": "VEHICULE", "pattern": [{"LOWER": "poids"}, {"LOWER": "lourd"}]},
    {"label": "VEHICULE", "pattern": [{"LOWER": "motocycle"}]},
    {"label": "VEHICULE", "pattern": [{"LOWER": "véhicule"}, {"LOWER": "automobile"}]},
    # Articles
    {"label": "ARTICLE_LOI", "pattern": [{"LOWER": "article"}, {"TEXT": {"REGEX": r"\d+"}}]},
]

ruler.add_patterns(patterns)

# Appliquer sur tout le dataframe
def extraire_entites(text):
    doc = nlp(text[:500])
    return [(ent.text, ent.label_) for ent in doc.ents]

df['entites_ner'] = df['infraction_desc'].apply(extraire_entites)

# Aperçu
print("NER results :")
for _, row in df[df['amende_min'].notna()].head(5).iterrows():
    print(f"Art.{row['article_id']} | {row['entites_ner']}")

NER results :
Art.149 | [('amende de', 'AMENDE')]
Art.361 | [('emprisonnement', 'PEINE'), ('amende de', 'AMENDE')]
Art.151 | [('emprisonnement', 'PEINE'), ('amende de', 'AMENDE')]
Art.152 | [('amende de', 'AMENDE')]
Art.154 | [('Pobtention d’', 'MISC'), ('d’', 'MISC'), ('emprisonnement', 'PEINE'), ('amende de', 'AMENDE')]


In [20]:
df.to_csv('export_final.csv', index=False, encoding='utf-8-sig')

print(" export_final.csv généré !")
print(f"Shape : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
print(f"\nArticles avec amendes : {df['amende_min'].notna().sum()}")
print(f"Clusters : {df['cluster'].nunique()}")
print(f"\nAperçu final :")
print(df[['article_id','amende_min','amende_max','type_infraction','niveau_gravite','mots_cles','cluster']].head(10).to_string())

 export_final.csv généré !
Shape : (86, 12)
Colonnes : ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_min', 'amende_max', 'points_retrait', 'classe_infraction', 'type_infraction', 'niveau_gravite', 'mots_cles', 'cluster', 'entites_ner']

Articles avec amendes : 34
Clusters : 5

Aperçu final :
   article_id  amende_min  amende_max type_infraction niveau_gravite                                                                                           mots_cles  cluster
0          95         NaN         NaN      infraction         faible                                                                                             général        1
1          38         NaN         NaN      infraction         faible                                                                                             général        3
2          96         NaN         NaN      infraction         faible                                                                                 permis

## TO DO
1. Compléter les fonctions incomplètes et les améliorer
2. Ajouter au moins 5 nouveaux patterns Regex
3. Ajouter de nouvelles colonnes :
   - type_infraction
   - niveau_gravite
   - localisation
   - categorie_vehicule
   - autres ....
4. Travailler sur le document donné
5. Comparer Rule-based vs BERT